In [1]:
# Carrega os dados já limpos e prontos de acordo com a análise exploratoria para serem usados

from src.preparo import carregar_particao

X_treino, X_teste, y_treino, y_teste = carregar_particao()

In [2]:
# Monta o modelo: escalonamento + SGD em um único Pipeline (ainda sem treinar)
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 2. Criar a estrutura do modelo com o escalonador necessário para o SGD
sgd_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SGDClassifier(loss='log_loss', random_state=42, max_iter=1000))
])

In [3]:
# Ajusta os hiperparâmetros do SGD com busca aleatória (métrica: Average Precision nos folds do treino)
# O scorer 'average_precision' ranqueia pela decision_function (o predict_proba do SGD satura em 1,0)
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

from src.preparo import SEED, cv_tuning

espaco = {
    'model__alpha': loguniform(1e-6, 1e-1),
    'model__penalty': ['l2', 'l1', 'elasticnet'],
    'model__class_weight': [None, 'balanced'],
}

busca = RandomizedSearchCV(sgd_pipeline, espaco, n_iter=30, scoring='average_precision', cv=cv_tuning(), random_state=SEED)
busca.fit(X_treino, y_treino)

# A busca já retreina o melhor pipeline com todo o treino
sgd_pipeline = busca.best_estimator_

print(busca.best_params_)
print(f'Melhor AP médio nos folds: {busca.best_score_:.4f}')

/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


... (17 outputs left out)

/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/usr/lib/python3/dist-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


{'model__alpha': np.float64(0.0010129197956845735), 'model__class_weight': 'balanced', 'model__penalty': 'elasticnet'}
Melhor AP médio nos folds: 0.7521


In [4]:
# Escolhe o limiar que maximiza o F2 nas previsões out-of-fold do treino (nunca 0,5 fixo)
# Cada transação é pontuada por um modelo que não a viu no treino; a pontuação é a decision_function
import numpy as np
from sklearn.metrics import average_precision_score, fbeta_score, precision_recall_curve
from sklearn.model_selection import cross_val_predict

from src.preparo import cv_tuning

pontuacao_oof = cross_val_predict(sgd_pipeline, X_treino, y_treino, cv=cv_tuning(), method='decision_function')
print(f'Average Precision (out-of-fold): {average_precision_score(y_treino, pontuacao_oof):.4f}')

precisao, recall, limiares = precision_recall_curve(y_treino, pontuacao_oof)

# precisao e recall têm um ponto a mais que limiares (o ponto final, sem limiar); descartamos esse
precisao, recall = precisao[:-1], recall[:-1]

# F2 = 5·P·R / (4·P + R); onde o denominador é zero, o F2 vale 0
denominador = 4 * precisao + recall
f2 = np.divide(5 * precisao * recall, denominador, out=np.zeros_like(denominador), where=denominador > 0)

melhor = np.argmax(f2)
limiar = limiares[melhor]

print(f'Limiar escolhido (escala da decision_function): {limiar:.4f}')
print(f'F2: {f2[melhor]:.4f} | precisão: {precisao[melhor]:.4f} | recall: {recall[melhor]:.4f}')
# decision_function >= 0 equivale a probabilidade >= 0,5 (o que o predict() faz)
print(f'F2 com limiar 0,5: {fbeta_score(y_treino, pontuacao_oof >= 0, beta=2):.4f}')

Average Precision (out-of-fold): 0.7411
Limiar escolhido (escala da decision_function): 12.9081
F2: 0.7960 | precisão: 0.7844 | recall: 0.7989
F2 com limiar 0,5: 0.1981


In [5]:
# Avaliação final no teste (usado uma única vez): decision_function do modelo ajustado + limiar escolhido no treino
from sklearn.metrics import roc_auc_score, average_precision_score, fbeta_score, confusion_matrix, precision_score, recall_score

# 1. Obter pontuações (decision_function, pois o predict_proba satura em 1,0) e predições com o limiar do treino
pontuacao_teste = sgd_pipeline.decision_function(X_teste)
y_pred = pontuacao_teste >= limiar

# 2. Calcular matriz de confusão
tn, fp, fn, tp = confusion_matrix(y_teste, y_pred).ravel()

# 3.métricas formatadas 
print(f"Average Precision: {average_precision_score(y_teste, pontuacao_teste):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_teste, pontuacao_teste):.4f}")
print(f"F2: {fbeta_score(y_teste, y_pred, beta=2):.4f}")
print(f"Precisão: {precision_score(y_teste, y_pred):.4f} | Recall: {recall_score(y_teste, y_pred):.4f}")
print(f"Verdadeiros negativos: {tn} | Falsos positivos: {fp} | Falsos negativos: {fn} | Verdadeiros positivos: {tp}")

Average Precision: 0.6866
ROC-AUC: 0.9640
F2: 0.7676
Precisão: 0.8090 | Recall: 0.7579
Verdadeiros negativos: 56634 | Falsos positivos: 17 | Falsos negativos: 23 | Verdadeiros positivos: 72
